# MediaLab · Proyecto significativo 4
**Lorena Abigail Coca Ccorpa · SENATI**

Gestión de publicidad, Ridge, K-Means y visión computacional. Semanas 10 y 11.

## Objetivo
Unir la gestión de campañas con estimaciones de precio y preparación de fotografías. Los datos son sintéticos y no corresponden a clientes reales.

## Cómo ejecutar
Instala requirements.txt desde la raíz del proyecto, abre este cuaderno y ejecuta las celdas en orden. No requiere cámara para el experimento reproducible.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import numpy as np
base = Path.cwd()
if not (base / "entrenamiento").exists():
    base = base.parent
sys.path.insert(0, str(base))
from entrenamiento.entrenar import train, ROOT, FEATURES
metricas = train()
metricas

## 1. Datos, variables y álgebra lineal
X es una matriz de campañas por características; y es un vector de precios. Los campos categóricos se codifican con OneHotEncoder y los numéricos se estandarizan. Esto permite operar con matrices y comparar distancias.

La IA es el campo amplio; Machine Learning aprende patrones a partir de ejemplos. Deep Learning usa redes neuronales profundas. Aquí se implementan métodos clásicos de ML, no una red profunda.

In [ ]:
df = pd.read_csv(ROOT / "datos_sinteticos.csv")
print("Dimensiones:", df.shape)
print("Nulos:", df.isna().sum().sum(), "Duplicados:", df.duplicated().sum())
df.head()

## 2. Estadística
La media resume el promedio; la mediana es el valor central. La varianza mide dispersión en unidades al cuadrado y la desviación estándar devuelve la unidad original (soles). Son descripciones del dataset artificial.

In [ ]:
precio = df.precio_soles
pd.Series({"media": precio.mean(), "mediana": precio.median(), "varianza": precio.var(), "desviacion": precio.std()})

## 3. Partición y regresión
Regresión simple emplea una variable de entrada; múltiple usa varias. Este proyecto implementa Ridge múltiple con penalización L2, por lo que no cuenta como una nueva implementación de regresión simple.

Se separa entrenamiento/prueba antes del ajuste. GridSearchCV compara alfa 0.1, 1, 5 y 20 en cinco pliegues del entrenamiento; el conjunto de prueba se usa al final.

In [ ]:
entrenamiento = pd.read_csv(ROOT / "entrenamiento.csv")
prueba = pd.read_csv(ROOT / "prueba.csv")
assert not set(entrenamiento.id_sintetico) & set(prueba.id_sintetico)
print("Entrenamiento:", len(entrenamiento), "Prueba:", len(prueba))
pd.DataFrame(metricas["cv"])

## 4. Evaluación independiente
MAE es error absoluto promedio; RMSE penaliza errores grandes; R² compara el ajuste con la variación de los precios. Un R² alto en datos sintéticos no garantiza resultados reales. La referencia predice siempre la media del entrenamiento.

In [ ]:
pd.Series({k: metricas[k] for k in ["mae", "rmse", "r2", "mae_referencia"]})

## 5. Agrupamiento K-Means
Clasificación supervisada necesita etiquetas conocidas, como categorías asignadas previamente. Agrupamiento no supervisado descubre similitudes sin etiquetas. K-Means asigna ejemplos al centro más cercano en el espacio estandarizado.

Se usan duración, emisiones, días y alcance, sin el precio objetivo. Los tres grupos se presentan por número para no inventar categorías comerciales. Silhouette cercano a cero sugiere separación limitada.

In [ ]:
pd.DataFrame(metricas["kmeans"]["centros"], columns=metricas["kmeans"]["features"]).assign(cantidad=metricas["kmeans"]["cantidades"])

In [ ]:
print("Silhouette:", metricas["kmeans"]["silhouette"])
import joblib
modelos = joblib.load(ROOT / "modelos.joblib")
ejemplo = prueba[FEATURES].iloc[[0]]
print("Precio:", modelos["ridge"].predict(ejemplo)[0])
print("Grupo:", modelos["kmeans"].predict(modelos["scaler"].transform(ejemplo[modelos["cluster_features"]]))[0]+1)

## 6. OpenCV: de la clase al estudio creativo
El sitio captura una imagen con la cámara del navegador y la envía, con autorización, a Flask. OpenCV aplica filtros en el servidor: gris, GaussianBlur, Canny, tintes o Haar Cascade para rostros. No se guarda la imagen en el servidor ni se transmite video continuo.

La detección localiza regiones que parecen rostros; no reconoce identidad ni determina emociones. El ejemplo de DeepFace recibido como material de clase es un antecedente independiente.

In [ ]:
import cv2
imagen = np.zeros((120, 160, 3), dtype=np.uint8)
imagen[:, 80:] = (50, 160, 240)
gris = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)
bordes = cv2.Canny(gris, 80, 160)
print("Luminosidad media:", gris.mean())
print("Píxeles de borde:", np.count_nonzero(bordes))

## 7. Prueba de integración
La API valida tipos, categorías y rangos. Este ejemplo usa el cliente de pruebas de Flask, sin publicar datos externos.

In [ ]:
from app import app
cliente = app.test_client()
respuesta = cliente.post("/api/cotizar", json=ejemplo.iloc[0].to_dict())
assert respuesta.status_code == 200
respuesta.get_json()

## 8. Manual breve y evidencia
1. Abre la web y crea una campaña de prueba.
2. Calcula una estimación y úsala en una campaña.
3. Activa cámara, captura y aplica filtro con autorización.
4. Guarda/descarga una foto.
5. Agrupa tus campañas y explica las características de cada centro.
6. Exporta un respaldo JSON y comprueba su restauración.
7. Revisa docs/OPERACIONES_PEA.md y marca solo lo demostrado.

Las campañas viven en localStorage y la galería en IndexedDB: no hay sincronización entre dispositivos. El respaldo JSON no contiene fotos.

## Referencias
OpenCV, documentación oficial: https://docs.opencv.org/4.x/
Scikit-learn, guía de usuario: https://scikit-learn.org/stable/
Flask, documentación: https://flask.palletsprojects.com/
Render: https://render.com/docs/free